# N2 · Preparing Variables

*R for Survey Data, lesson 2 of 7 · about 30 to 40 minutes*

**In this notebook you will learn to:**

- Turn special codes into missing values (`NA`)
- Build the analysis variables: fair or poor health, cancer survivor, income, and education
- Check every new variable against the original codes
- Put categories in a sensible order with `factor()`

**Before this notebook:** [N1: Meet the NHIS](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/N1_Meet_the_NHIS.ipynb).

> **Before you start:** click **File > Save a copy in Drive** and work in your copy. Run a code cell by clicking the play button to its left, or press **Shift + Enter**. Go in order from top to bottom.

## 1. The plan

In N1 you met the raw NHIS codes. Before any analysis, they need to become variables that answer the research question:

| New variable | Built from | Meaning |
|---|---|---|
| `fair_poor` | `PHSTAT_A` | 1 = fair or poor health, 0 = good, very good, or excellent |
| `survivor` | `CANEV_A`, `NUMCAN_A`, `SKNNMCAN_A` | 1 = has had cancer, 0 = has not |
| `poverty` | `POVRATTC_A` | Four income groups |
| `education` | `EDUCP_A` | Three education groups |
| `age_group`, `sex`, `insured` | `AGEP_A`, `SEX_A`, `NOTCOV_A` | Characteristics to describe and adjust for |

Two rules for the whole lesson:

1. **Never overwrite an original column.** Create a new one, so you can always compare old and new.
2. **Keep every row.** Even though the question is about cancer survivors, do not remove anyone. Lesson N5 explains why this matters for survey data.

Run the setup cell. The lines printed by `library(tidyverse)` and `read_csv()` are normal.

In [ ]:
library(tidyverse)

nhis <- read_csv("https://raw.githubusercontent.com/icohr-lab/tutorials/refs/heads/main/data/nhis/nhis2025_adult.csv")

## 2. Special codes to `NA`

Start with self-rated health. Codes 7, 8, and 9 are not answers, so they should become `NA`. `if_else()` does it in one line: if the code is a special code, use `NA`; otherwise keep the code.

In [ ]:
nhis <- nhis %>%
  mutate(health = if_else(PHSTAT_A %in% c(7, 8, 9), NA, PHSTAT_A))

nhis %>% count(PHSTAT_A, health)

Counting the old and new columns together shows exactly what happened: codes 1 to 5 kept their values, and the 7 people with codes 7 and 9 now have `NA`. Code 8 is included even though nobody has it this year, because the codebook lists it.

## 3. The outcome: fair or poor health

Researchers usually split self-rated health into two groups: **fair or poor** versus **good, very good, or excellent**. This is the outcome for the project.

Here `case_when()` uses `.default = NA` on purpose: anything that is not a real answer becomes missing.

**Your turn 1.** Create `fair_poor`: 1 when `PHSTAT_A` is 4 or 5, 0 when it is 1, 2, or 3, and `NA` otherwise. Store it in `nhis`, then check it with `count(PHSTAT_A, fair_poor)`.

*Expected result: 3,815 people with 1 (2,970 + 845), 20,393 with 0, and 7 with `NA`*

In [ ]:
# Your code here

## 4. Who counts as a cancer survivor?

`CANEV_A` asks whether a doctor ever said you had cancer. One adjustment is standard in cancer survivorship research: people whose **only** cancer was **non-melanoma skin cancer** are not counted as survivors. These skin cancers are very common and rarely life-threatening, and including them would mix many people with a very different experience into the survivor group.

So a survivor is someone who answered Yes to `CANEV_A`, unless their only cancer type was non-melanoma skin cancer: `NUMCAN_A` is 1 and `SKNNMCAN_A` is 1.

### A trap with `==` and `NA`

`SKNNMCAN_A` is `NA` for everyone who was never asked. Watch what `==` does with `NA`.

In [ ]:
x <- c(1, 2, NA)
x == 1
x %in% 1

`NA == 1` gives `NA` ("unknown"), but `NA %in% 1` gives `FALSE`. For "is this value exactly 1?", `%in%` is the safer choice when a column has missing values.

In [ ]:
nhis <- nhis %>%
  mutate(
    skin_only = NUMCAN_A == 1 & SKNNMCAN_A %in% 1,
    survivor = case_when(
      CANEV_A == 1 & !skin_only ~ 1,
      CANEV_A == 2 | skin_only ~ 0,
      .default = NA
    )
  )

nhis %>% count(CANEV_A, skin_only, survivor)

Read the rules: `!` means "not" and `|` means "or". A survivor said Yes and is **not** skin-only. A non-survivor said No **or** is skin-only. Refused and don't know (7 and 9) become `NA`.

The check shows 545 people whose only cancer was non-melanoma skin cancer moved from Yes to 0.

**Your turn 2.** How many survivors are there, how many non-survivors, and how many `NA`? Use `count(survivor)`.

*Expected result: 2,833 survivors, 21,349 non-survivors, 33 `NA`*

In [ ]:
# Your code here

## 5. Income groups

`POVRATTC_A` is a number, but for comparing groups it helps to cut it into categories. A common choice uses multiples of the poverty line: below 100%, 100 to 199%, 200 to 399%, and 400% or more.

`case_when()` checks its rules **in order** and uses the first one that is true. So the second rule, `POVRATTC_A < 2`, only ever sees people who failed the first rule, meaning people at 1 or above.

**Your turn 3.** Create `poverty` with these four groups, as text: "Below 100%", "100-199%", "200-399%", "400% or more". Store it in `nhis` and count it.

*Expected result: Below 100%: 2,211; 100-199%: 4,181; 200-399%: 7,300; 400% or more: 10,523; no `NA`*

In [ ]:
# Your code here

## 6. Your turn: education groups

Combine the ten education codes into three groups, using the code table from N1:

| Codes | Group |
|---|---|
| 1 to 4 | High school or less |
| 5 to 7 | Some college (including associate degrees) |
| 8 to 10 | Bachelor or higher |
| 97, 98, 99 | `NA` |

**Your turn 4.** Create `education` with the groups "High school or less", "Some college", and "Bachelor or higher". Check it with `count(EDUCP_A, education)` and then `count(education)`.

*Expected result: Bachelor or higher 9,577; High school or less 7,900; Some college 6,655; `NA` 83*

In [ ]:
# Your code here

## 7. The other variables

Age, sex, and insurance follow the same pattern. Run this cell, then read it line by line: every piece is something you have already used.

In [ ]:
nhis <- nhis %>%
  mutate(
    age = if_else(AGEP_A %in% c(97, 98, 99), NA, AGEP_A),
    age_group = case_when(
      age < 45 ~ "18-44",
      age < 65 ~ "45-64",
      age >= 65 ~ "65+",
      .default = NA
    ),
    sex = case_when(SEX_A == 1 ~ "Male", SEX_A == 2 ~ "Female", .default = NA),
    insured = case_when(NOTCOV_A == 2 ~ "Yes", NOTCOV_A == 1 ~ "No", .default = NA)
  )

Remember that `age` is still top-coded: 85 means 85 or older. That does not matter for `age_group`, since everyone 85 and over belongs in "65+" anyway.

**Your turn 5.** Check the three new variables with `count()`: `age_group`, `sex`, and `insured`.

*Expected result: age_group: 18-44 8,485; 45-64 7,202; 65+ 8,468; `NA` 60. sex: Female 12,975; Male 11,215; `NA` 25. insured: Yes 22,520; No 1,626; `NA` 69*

In [ ]:
# Your code here

## 8. Putting categories in order

`count()` lists text categories alphabetically, which is why "100-199%" came before "Below 100%". Tables and plots read much better in a natural order. `factor()` with `levels` sets the order.

In [ ]:
nhis <- nhis %>%
  mutate(
    poverty = factor(poverty, levels = c("Below 100%", "100-199%", "200-399%", "400% or more")),
    education = factor(education, levels = c("High school or less", "Some college", "Bachelor or higher"))
  )

nhis %>% count(poverty)

Same counts, now in order from lowest to highest income.

## 9. The analysis file

Now keep just the columns you need, **including the three survey design columns**. Without `WTFA_A`, `PSTRAT`, and `PPSU`, the analyses in N4 to N6 are impossible.

In [ ]:
analysis <- nhis %>%
  select(HHX, WTFA_A, PSTRAT, PPSU,
         fair_poor, survivor, poverty, education, age, age_group, sex, insured)

dim(analysis)

24,215 rows: everyone is still here. 12 columns.

## 10. A first look, and a warning

Here is the share of survivors and non-survivors in fair or poor health. `mean()` of a 0/1 column is the proportion of 1s.

In [ ]:
analysis %>%
  filter(!is.na(survivor), !is.na(fair_poor)) %>%
  group_by(survivor) %>%
  summarize(people = n(), pct_fair_poor = 100 * mean(fair_poor))

About 27% of survivors report fair or poor health, compared with about 14% of everyone else.

**Your turn 6.** Among **survivors only**, calculate the percent in fair or poor health for each education group. Leave out people with missing education or missing `fair_poor`.

*Expected result: High school or less 39.7% (867 people); Some college 28.4% (846); Bachelor or higher 15.9% (1,109)*

In [ ]:
# Your code here

A striking gradient. But **these numbers are not yet results.** They describe the people who happened to be interviewed, without the weights. Not everyone had the same chance of being interviewed, so unweighted percentages can differ from the true U.S. figures. Lesson N4 shows how much, and how to fix it.

## 11. Save your work

Colab's memory is wiped when your session ends. `write_csv()` saves the analysis file, and Colab's **Files** panel (the folder icon on the left) lets you download it.

In [ ]:
write_csv(analysis, "nhis_analysis.csv")

In the next lesson you move to RStudio on your own computer, where files stay put, and you will run the code from this lesson as a script.

---
## Recap

| Code | What it does |
|---|---|
| `if_else(x %in% c(7, 8, 9), NA, x)` | Turn special codes into `NA` |
| `case_when(..., .default = NA)` | Anything that matches no rule becomes `NA` |
| `x %in% 1` | Like `x == 1`, but `FALSE` instead of `NA` when x is missing |
| `!a, a | b` | Not a; a or b |
| `count(old, new)` | Check a recode against the original codes |
| `factor(x, levels = c(...))` | Put categories in a set order |
| `100 * mean(x)` | Percent of 1s in a 0/1 column |

---
## Solutions

Try each exercise before looking. There is often more than one correct way to write it.

**Your turn 1**

```r
nhis <- nhis %>%
  mutate(fair_poor = case_when(
    PHSTAT_A %in% c(4, 5) ~ 1,
    PHSTAT_A %in% c(1, 2, 3) ~ 0,
    .default = NA
  ))

nhis %>% count(PHSTAT_A, fair_poor)
```

**Your turn 2**

```r
nhis %>% count(survivor)
```

**Your turn 3**

`POVRATTC_A` has no missing values (they were imputed), so no `NA` appears.

```r
nhis <- nhis %>%
  mutate(poverty = case_when(
    POVRATTC_A < 1 ~ "Below 100%",
    POVRATTC_A < 2 ~ "100-199%",
    POVRATTC_A < 4 ~ "200-399%",
    POVRATTC_A >= 4 ~ "400% or more"
  ))

nhis %>% count(poverty)
```

**Your turn 4**

```r
nhis <- nhis %>%
  mutate(education = case_when(
    EDUCP_A %in% 1:4 ~ "High school or less",
    EDUCP_A %in% 5:7 ~ "Some college",
    EDUCP_A %in% 8:10 ~ "Bachelor or higher",
    .default = NA
  ))

nhis %>% count(EDUCP_A, education)
nhis %>% count(education)
```

**Your turn 5**

```r
nhis %>% count(age_group)
nhis %>% count(sex)
nhis %>% count(insured)
```

**Your turn 6**

```r
analysis %>%
  filter(survivor == 1, !is.na(fair_poor), !is.na(education)) %>%
  group_by(education) %>%
  summarize(people = n(), pct_fair_poor = 100 * mean(fair_poor))
```

---
## Next

Continue with [N3: Moving to RStudio](https://icohr-lab.github.io/tutorials/survey/n3-rstudio.html), a lesson on the tutorials website.